# multiply-back — worked example 3: Verify multiply_back shapes with (3,1,5) x (1,4,1) broadcasting

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `multiply-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The backward pass for element-wise multiplication must return a gradient with exactly the same shape as the corresponding forward-pass input. When inputs have different shapes that broadcast to a larger output, `unbroadcast` must identify and sum over the expanded dimensions. Verifying shapes explicitly before any numerical check is a good debugging habit.

## Worked solution

**Step 1 — set up the forward pass with two non-trivially-shaped tensors.** We use `x: (3, 1, 5)` and `y: (1, 4, 1)`. The broadcast output has shape `(3, 4, 5)`.

**Step 2 — create a gradient tensor of the output shape.** `grad_out: (3, 4, 5)` — same shape as `out`.

**Step 3 — apply multiply_back0 for x.** `raw = grad_out * y` has shape `(3, 4, 5)`. The `unbroadcast` call must collapse dim=1 (which was size 1 in x) back to size 1, giving `(3, 1, 5)`.

**Step 4 — apply multiply_back1 for y.** `raw = grad_out * x` has shape `(3, 4, 5)`. `unbroadcast` must collapse dim=0 (size 1 in y) and dim=2 (size 1 in y), giving `(1, 4, 1)`.

**Step 5 — compare to autograd.** Run the same computation with `requires_grad=True` and `backward()`, then assert closeness.

In [ ]:
import torch
from torch import Tensor

torch.manual_seed(0)

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out, out, x, y):
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out, out, x, y):
    return unbroadcast(grad_out * x, y)

# Exercise: shapes (3,1,5) and (1,4,1)
torch.manual_seed(8)
x = torch.randn(3, 1, 5, requires_grad=True)
y = torch.randn(1, 4, 1, requires_grad=True)
out = x * y  # (3, 4, 5)
loss = out.sum()
loss.backward()

# Hand-written backward
grad_out = torch.ones(3, 4, 5)
gx = multiply_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = multiply_back1(grad_out, out.detach(), x.detach(), y.detach())

print(f"x shape: (3,1,5), gx shape: {gx.shape}")  # should be (3,1,5)
print(f"y shape: (1,4,1), gy shape: {gy.shape}")  # should be (1,4,1)
assert gx.shape == (3, 1, 5), f"wrong shape: {gx.shape}"
assert gy.shape == (1, 4, 1), f"wrong shape: {gy.shape}"
assert torch.allclose(gx, x.grad, atol=1e-6)
assert torch.allclose(gy, y.grad, atol=1e-6)
print("3-D multiply_back shapes and values correct!")